# Exercise 2.2.1.3 — read & understand other probe environments

> Part of [Delta Drills](https://delta-drills.vercel.app) ARENA practice. When the test cell passes, your completion is reported back to your account automatically.

**Section:** `2.2.1 DQN`  
**Notebook:** `2.2.1_Deep_Q_Networks_exercises.ipynb`  
**Return to Delta Drills:** [https://delta-drills.vercel.app/?arena_exercise=2.2.1.3](https://delta-drills.vercel.app/?arena_exercise=2.2.1.3)


# [2.2.1] - Deep Q Networks (exercises)

> **ARENA [Streamlit Page](https://arena-chapter2-rl.streamlit.app/20_[2.2.1]_Deep_Q_Networks)**
>
> **Colab: [exercises](https://colab.research.google.com/github/callummcdougall/ARENA_3.0/blob/main/chapter2_rl/exercises/part21_dqn/2.2.1_Deep_Q_Networks_exercises.ipynb?t=20250917) | [solutions](https://colab.research.google.com/github/callummcdougall/ARENA_3.0/blob/main/chapter2_rl/exercises/part21_dqn/2.2.1_Deep_Q_Networks_solutions.ipynb?t=20250917)**

Please send any problems / bugs on the `#errata` channel in the [Slack group](https://join.slack.com/t/arena-uk/shared_invite/zt-3afdmdhye-Mdb3Sv~ss_V_mEaXEbkABA), and ask any questions on the dedicated channels for this chapter of material.

You can collapse each section so only the headers are visible, by clicking the arrow symbol on the left hand side of the markdown header cells.

Links to all other chapters: [(0) Fundamentals](https://arena-chapter0-fundamentals.streamlit.app/), [(1) Transformer Interpretability](https://arena-chapter1-transformer-interp.streamlit.app/), [(2) RL](https://arena-chapter2-rl.streamlit.app/).

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/headers/header-22.png" width="350">

# Introduction

In this section, you'll implement Deep Q-Learning, often referred to as DQN for "Deep Q-Network". This was used in a landmark paper [Playing Atari with Deep Reinforcement Learning](https://www.cs.toronto.edu/~vmnih/docs/dqn.pdf). 

<!-- 
You'll also implement Vanilla Policy Gradient (VPG), the first policy gradient algorithm upon which many modern RL algorithms are based (including PPO).
This was the first working version of deep learning in the RL setting.

At the time, the idea that convolutional neural networks could look at Atari game pixels and "see" gameplay-relevant features like a Space Invader was new and noteworthy. In 2022, we take for granted that convnets work, so we're going to focus on the RL aspect and not the vision aspect today. -->

## Content & Learning Objectives


### 1️⃣ DQN

In this section, you'll implement Deep Q-Learning, often referred to as DQN for "Deep Q-Network". This was used in a landmark paper Playing Atari with [Deep Reinforcement Learning](https://www.cs.toronto.edu/~vmnih/docs/dqn.pdf).

You'll apply the technique of DQN to master the famous CartPole environment (below), and then (if you have time) move on to harder challenges like Acrobot and MountainCar.

> ##### Learning Objectives
>
> - Understand the DQN algorithm
> - Learn more about RL debugging, and build probe environments to debug your agents
> - Create a replay buffer to store environment transitions
> - Implement DQN using PyTorch, on the CartPole environment

## Setup (don't read, just run!)

In [ ]:
from __future__ import annotations

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

chapter = "chapter2_rl"
repo = "ARENA_3.0"
branch = "main"

# Install dependencies
try:
    import jaxtyping
except:
    %pip install wandb==0.18.7 einops "gymnasium[atari, accept-rom-license, other]==0.29.0" pygame jaxtyping

# Get root directory, handling 3 different cases: (1) Colab, (2) notebook not in ARENA repo, (3) notebook in ARENA repo
root = (
    "/content"
    if IN_COLAB
    else "/root"
    if repo not in os.getcwd()
    else str(next(p for p in Path.cwd().parents if p.name == repo))
)

if Path(root).exists() and not Path(f"{root}/{chapter}").exists():
    if not IN_COLAB:
        !sudo apt-get install unzip
        %pip install jupyter ipython --upgrade

    if not os.path.exists(f"{root}/{chapter}"):
        !wget -P {root} https://github.com/callummcdougall/ARENA_3.0/archive/refs/heads/{branch}.zip
        !unzip {root}/{branch}.zip '{repo}-{branch}/{chapter}/exercises/*' -d {root}
        !mv {root}/{repo}-{branch}/{chapter} {root}/{chapter}
        !rm {root}/{branch}.zip
        !rmdir {root}/{repo}-{branch}


if f"{root}/{chapter}/exercises" not in sys.path:
    sys.path.append(f"{root}/{chapter}/exercises")

os.chdir(f"{root}/{chapter}/exercises")

In [ ]:
import os
import sys
import time
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import TypeAlias

import gymnasium as gym
import numpy as np
import torch as t
import wandb
from gymnasium.spaces import Box, Discrete
from jaxtyping import Bool, Float, Int
from torch import Tensor, nn
from tqdm import tqdm, trange

warnings.filterwarnings("ignore")

Arr = np.ndarray

# Make sure exercises are in the path
chapter = "chapter2_rl"
section = "part21_dqn"
root_dir = next(p for p in Path.cwd().parents if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

import part21_dqn.tests as tests
import part21_dqn.utils as utils
from part1_intro_to_rl.solutions import Environment, Norvig, Toy, find_optimal_policy
from part1_intro_to_rl.utils import set_global_seeds
from part21_dqn.utils import make_env
from plotly_utils import cliffwalk_imshow, line, plot_cartpole_obs_and_dones
from rl_utils import generate_and_plot_trajectory

device = t.device(
    "mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu"
)

# 1️⃣ Deep Q-Learning

> ##### Learning Objectives
>
> - Understand the DQN algorithm
> - Learn more about RL debugging, and build probe environments to debug your agents
> - Create a replay buffer to store environment transitions
> - Implement DQN using PyTorch, on the CartPole environment

In this section, you'll implement Deep Q-Learning, often referred to as DQN for "Deep Q-Network". This was used in a landmark paper [Playing Atari with Deep Reinforcement Learning](https://www.cs.toronto.edu/~vmnih/docs/dqn.pdf).

At the time, the paper was very exciting: The agent would play the game by only looking at the same screen pixel data that a human player would be looking at, rather than a description of where the enemies in the game world are. The idea that convolutional neural networks could look at Atari game pixels and "see" gameplay-relevant features like a Space Invader was new and noteworthy. In 2022, we take for granted that convnets work, so we're going to focus on the RL aspect solely, and not the vision component.

## Optional Readings

* [Deep Q Networks Explained](https://www.lesswrong.com/posts/kyvCNgx9oAwJCuevo/deep-q-networks-explained) (25 minutes)
    * A high-level distillation as to how DQN works.
    * Read sections 1-4 (further sections optional).
* [Andy Jones - Debugging RL, Without the Agonizing Pain](https://andyljones.com/posts/rl-debugging.html) (10 minutes)
    * Useful tips for debugging your code when it's not working.
    * Read up to (not including) the Common Fixes section. Also read the Practical Advice section, up to and including "Use probe agents". The rest of the post is optional, and you're recommended to come back to it near the end if you're stuck.
    * The "probe environments" (a collection of simple environments of increasing complexity) section will be our first line of defense against bugs, you'll implement these in exercises below.

### Interesting Resources (not required reading)

- [An Outsider's Tour of Reinforcement Learning](http://www.argmin.net/2018/06/25/outsider-rl/) - comparison of RL techniques with the engineering discipline of control theory.
- [Towards Characterizing Divergence in Deep Q-Learning](https://arxiv.org/pdf/1903.08894.pdf) - analysis of what causes learning to diverge
- [Divergence in Deep Q-Learning: Tips and Tricks](https://amanhussain.com/post/divergence-deep-q-learning/) - includes some plots of average returns for comparison
- [Deep RL Bootcamp](https://sites.google.com/view/deep-rl-bootcamp/lectures) - 2017 bootcamp with video and slides. Good if you like videos.
- [DQN debugging using OpenAI gym Cartpole](https://adgefficiency.com/dqn-debugging/) - random dude's adventures in trying to get it to work.
- [CleanRL DQN](https://github.com/vwxyzjn/cleanrl) - single file implementations of RL algorithms. Your starter code today is based on this; try not to spoiler yourself by looking at the solutions too early!
- [Deep Reinforcement Learning Doesn't Work Yet](https://www.alexirpan.com/2018/02/14/rl-hard.html) - 2018 article describing difficulties preventing industrial adoption of RL.
- [Deep Reinforcement Learning Works - Now What?](https://tesslerc.github.io/posts/drl_works_now_what/) - 2020 response to the previous article highlighting recent progress.
- [Seed RL](https://github.com/google-research/seed_rl) - example of distributed RL using Docker and GCP.

### Conceptual overview of DQN

DQN is the natural extension of Q-Learning into the domain of deep learning. The main difference is that, instead of a table to store all the Q-values for each state-action pair, we train a neural network to learn this function for us. The usual implementation (which we'll use here) is for the Q-network to take the state as input, and output a vector of optimalQ-values for each action, i.e. we're learning the function:

$$
s \to (Q^*(s, a_1), ..., Q^*(s, a_n))
$$

Below is an algorithm showing the conceptual overview of DQN. We cycle through the following process:

* Generate a batch of experiences using our current policy, by **epsilon-greedy sampling** (i.e. we mostly take the action with the highest Q-value, but occasionally take a random action to encourage exploration). Store these experiences in the **replay buffer**.
* Use these values to calculate a **TD (temporal difference) error**, and update our network.
    * To increase stability, we also have a **target network** we use for the "next step" part of the TD error. This is a lagged copy of the Q-network (i.e. we update our Q-network via gradient descent, and then every so often we copy the Q-network weights over to our target network).
* Repeat this until convergence.

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/ppo-alg-conceptual-2.png" width="750">

### Fast Feedback Loops

We want to have faster feedback loops, and learning from Atari pixels doesn't achieve that. It might take 15 minutes per training run to get an agent to do well on Breakout, and that's if your implementation is relatively optimized. Even waiting 5 minutes to learn Pong from pixels is going to limit your ability to iterate, compared to using environments that are as simple as possible.

### CartPole

The classic environment "CartPole-v1" is simple to understand, yet hard enough for a RL agent to be interesting, by the end of the day your agent will be able to do this and more! (Click to watch!)


[![CartPole](https://img.youtube.com/vi/46wjA6dqxOM/0.jpg)](https://www.youtube.com/watch?v=46wjA6dqxOM "CartPole")

If you'd like to try the CartPole environment yourself, [click here to open the simulation](https://davidquarel.github.io/cartpole) in a new tab. 
* Use Left/Right arrow keys to move the cart, 
* R to reset, 
* Q to quit. 
* Use F/S to make the simulation faster/slower. 

Unlike the real CartPole environment, this simulation will not terminate the episode if the pole falls over.
We've also cheated here and added a hidden third no-op action, such that if no button is pressed, no force is applied to the cart.
This makes the simulation a bit easier for you as the human to play.
The real cartpole environment doesn't act like this: the agent must choose to push the cart either left or right on each timestep.

The description of the task is [here](https://gymnasium.farama.org/environments/classic_control/cart_pole/). Note that unlike the previous environments, the observation here is now continuous. You can see the source for CartPole [here](https://github.com/Farama-Foundation/Gymnasium/blob/v0.29.0/gymnasium/envs/classic_control/cartpole.py); don't worry about the implementation but do read the documentation to understand the format of the actions and observations.

The simple physics involved would be very easy for a model-based algorithm to fit, (this is a common assignment in control theory using [proportional-integral-derivative](https://en.wikipedia.org/wiki/PID_controller) (PID) controllers) but today we're doing it model-free: your agent has no idea that these observations represent positions or velocities, and it has no idea what the laws of physics are. The network has to learn in which direction to bump the cart in response to the current state of the world.

Each environment can have different versions registered to it. By consulting [the Gym source](https://github.com/Farama-Foundation/Gymnasium/blob/v0.29.0/gymnasium/envs/__init__.py) you can see that CartPole-v0 and CartPole-v1 are the same environment, except that v1 has longer episodes. Again, a minor change like this can affect what algorithms score well; an agent might consistently survive for 200 steps in an unstable fashion that means it would fall over if ran for 500 steps.

In [ ]:
env = gym.make("CartPole-v1", render_mode="rgb_array")

print(env.action_space)  # 2 actions: left and right
print(env.observation_space)  # Box(4): each action can take a continuous range of values

### Outline of the Exercises

The exercises are roughly split into 4 sections:

1. Implement the Q-network that maps a state to an estimated value for each action.
2. Implement a replay buffer to store experiences $e_t = (s_t, a_t, r_{t+1}, d_{t+1}, s_{t+1})$.
3. Implement the policy which chooses actions based on the Q-network, plus epsilon greedy randomness to encourage exploration.
4. Piece everything together into a training loop and train your agent.

## The Q-Network

The Q-Network takes in an observation $s$ and outputs a vector $[Q^*(s, a^1), \ldots Q^*(s,a^n)]$ representing an estimate of the optimal Q-value for the given state $s$, and each possible action $\mathcal{A} = \{a^1, \ldots, a^n\}$. This replaces our Q-value table used in Q-learning.

For best results, the architecture of the Q-network can be customized to each particular problem. For example, [the architecture of OpenAI Five](https://cdn.openai.com/research-covers/openai-five/network-architecture.pdf) used to play DOTA 2 is pretty complex and involves LSTMs.

For learning from pixels, a simple convolutional network and some fully connected layers does quite well. Where we have already processed features here, it's even easier: an MLP of this size should be plenty large for any environment today.

Implement the Q-network using a standard MLP, constructed of alternating Linear and ReLU layers.
The size of the input will match the dimensionality of the observation space, and the size of the output will match the number of actions to choose from (associating a reward to each.)
The dimensions of the hidden_sizes are provided.

Here is a diagram of what our particular Q-Network will look like for CartPole (you can open it in a new tab if it's hard to see clearly):

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/mermaid-diagram-2024-11-28-200952.svg" style="background-color: #fffbe0; padding: 10px" width="160"><br>

<!-- 
flowchart TD
    A["Input (num_obs,)"] -> B["Linear(num_obs, 120)"]
    B -> C[ReLU]
    C -> D["Linear(120, 84)"]
    D -> E[ReLU]
    E -> F["Linear(84, num_actions)"]
    F -> G["Output (num_actions,)"]
{
  "theme": "default",
  "themeVariables": {
    "fontSize": "22px"
  }
}
-->

<details>
<summary>Question - why do we not include a ReLU at the end?</summary>

If you end with a ReLU, then your network can only predict 0 or positive Q-values. This will cause problems as soon as you encounter an environment with negative rewards, or you try to do some scaling of the rewards.

</details>

<details>
<summary>Question - since CartPole-v1 gives +1 reward on every timestep, why do you think the network doesn't just learn the constant +1 function regardless of observation?</summary>

The network is learning Q-values (the sum of all future expected discounted rewards from this state/action pair), not rewards. Correspondingly, once the agent has learned a good policy, the Q-value associated with state action pair (pole is slightly left of vertical, move cart left) should be large, as we would expect a long episode (and correspondingly lots of reward) by taking actions to help to balance the pole. Pairs like (cart near right boundary, move cart right) cause the episode to terminate, and as such the network will learn low Q-values.

</details>

## Connect to Delta Drills

Paste your Delta Drills auth token below so this exercise can report its completion back to your account.
You can copy the token from your Delta Drills account page.


In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_EXERCISE_ID = "2.2.1.3"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"


### Prior-exercise solutions (auto-imported)

These were imported from ARENA's reference `solutions.py` so you can jump straight into this exercise without having implemented every predecessor. Re-implement them yourself if you'd rather build top-to-bottom.


In [ ]:
from part21_dqn.solutions import QNetwork, linear_schedule, epsilon_greedy_policy


### Exercise - read & understand other probe environments

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵🔵⚪
> 
> You should spend up to 10-20 minutes here.
> ```

For each of the probes below, read their implementation code, and understand how they correspond to their docstrings (and to the [descriptions](https://andyljones.com/posts/rl-debugging.html#:~:text=Use%20probe%20environments) given in Andy Jones' post).

It's very important to understand how these probes work, and why they're useful tools for debugging. When you're working on your own RL projects, you might have to write your own probes to suit your particular use cases.

In [ ]:
class Probe2(gym.Env):
    """
    One action, observation of [-1.0] or [+1.0], one timestep long, reward equals observation.

    We expect the agent to rapidly learn the value of each observation is equal to the observation.
    """

    action_space: Discrete
    observation_space: Box

    def __init__(self, render_mode: str = "rgb_array"):
        super().__init__()
        self.observation_space = Box(np.array([-1.0]), np.array([+1.0]))
        self.action_space = Discrete(1)
        self.reset()
        self.reward = None

    def step(self, action: ActType) -> tuple[ObsType, float, bool, bool, dict]:
        assert self.reward is not None
        return np.array([self.observation]), self.reward, True, True, {}

    def reset(self, seed: int | None = None, options=None) -> ObsType | tuple[ObsType, dict]:
        super().reset(seed=seed)
        self.reward = 1.0 if self.np_random.random() < 0.5 else -1.0
        self.observation = self.reward
        return np.array([self.reward]), {}


class Probe3(gym.Env):
    """
    One action, [0.0] then [1.0] observation, two timesteps, +1 reward at the end.

    We expect the agent to rapidly learn the discounted value of the initial observation.
    """

    action_space: Discrete
    observation_space: Box

    def __init__(self, render_mode: str = "rgb_array"):
        super().__init__()
        self.observation_space = Box(np.array([-0.0]), np.array([+1.0]))
        self.action_space = Discrete(1)
        self.reset()

    def step(self, action: ActType) -> tuple[ObsType, float, bool, bool, dict]:
        self.n += 1
        if self.n == 1:
            return np.array([1.0]), 0.0, False, False, {}
        elif self.n == 2:
            return np.array([0.0]), 1.0, True, True, {}
        raise ValueError(self.n)

    def reset(self, seed: int | None = None, options=None) -> ObsType | tuple[ObsType, dict]:
        super().reset(seed=seed)
        self.n = 0
        return np.array([0.0]), {}


class Probe4(gym.Env):
    """
    Two actions, [0.0] observation, one timestep, reward is -1.0 or +1.0 dependent on the action.

    We expect the agent to learn to choose the +1.0 action.
    """

    action_space: Discrete
    observation_space: Box

    def __init__(self, render_mode: str = "rgb_array"):
        self.observation_space = Box(np.array([-0.0]), np.array([+0.0]))
        self.action_space = Discrete(2)
        self.reset()

    def step(self, action: ActType) -> tuple[ObsType, float, bool, bool, dict]:
        reward = -1.0 if action == 0 else 1.0
        return np.array([0.0]), reward, True, True, {}

    def reset(self, seed: int | None = None, options=None) -> ObsType | tuple[ObsType, dict]:
        super().reset(seed=seed)
        return np.array([0.0]), {}


class Probe5(gym.Env):
    """
    Two actions, random 0/1 observation, one timestep, reward is 1 if action equals observation,
    otherwise -1.

    We expect the agent to learn to match its action to the observation.
    """

    action_space: Discrete
    observation_space: Box

    def __init__(self, render_mode: str = "rgb_array"):
        self.observation_space = Box(np.array([-1.0]), np.array([+1.0]))
        self.action_space = Discrete(2)
        self.reset()

    def step(self, action: ActType) -> tuple[ObsType, float, bool, bool, dict]:
        reward = 1.0 if action == self.obs else -1.0
        return np.array([self.obs]), reward, True, True, {}

    def reset(self, seed: int | None = None, options=None) -> ObsType | tuple[ObsType, dict]:
        super().reset(seed=seed)
        self.obs = 1.0 if self.np_random.random() < 0.5 else 0.0
        return np.array([self.obs], dtype=float), {}


gym.envs.registration.register(id="Probe2-v0", entry_point=Probe2)
gym.envs.registration.register(id="Probe3-v0", entry_point=Probe3)
gym.envs.registration.register(id="Probe4-v0", entry_point=Probe4)
gym.envs.registration.register(id="Probe5-v0", entry_point=Probe5)

A brief summary of these, along with recommendations of where to go to debug if one of them fails (note that these won't be true 100% of the time, but should hopefully give you some useful direction):

<details>
<summary>Summary of probes</summary>

1. **Tests basic learning ability**. If this fails, it means the agent has failed to learn to associate a constant observation with a constant reward. You should check your loss functions and optimizers in this case.
2. **Tests the agent's ability to differentiate between 2 different observations (and learn their respective values)**. If this fails, it means the agent has issues with handling multiple possible observations.
3. **Tests the agent's ability to handle time & reward delay**. If this fails, it means the agent has problems with multi-step scenarios of discounting future rewards. You should look at how your agent step function works.
4. **Tests the agent's ability to learn from actions leading to different rewards**. If this fails, it means the agent has failed to change its policy for different rewards, and you should look closer at how your agent is updating its policy based on the rewards it receives & the loss function.
5. **Tests the agent's ability to map observations to actions**. If this fails, you should look at the code which handles multiple timesteps, as well as the code that handles the agent's map from observations to actions.

</details>

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

def _dd_report_complete():
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    try:
        body = _dd_json.dumps({
            'exercise_id': DD_EXERCISE_ID,
            'passed': True,
        }).encode('utf-8')
        req = _dd_req.Request(
            f'{DD_BACKEND_URL}/api/arena/complete',
            data=body,
            headers={
                'Content-Type': 'application/json',
                'Authorization': f'Bearer {DD_TOKEN}',
            },
            method='POST',
        )
        with _dd_req.urlopen(req, timeout=3) as r:
            r.read()
        print(f'[Delta Drills] reported completion of {DD_EXERCISE_ID}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

# This exercise has no automatic test — call `_dd_report_complete()`
# in a new cell once you're satisfied with your answer.
